# LangGraph G12 — A supervisor and specialists
As CampusAI grows, one agent with every tool and one long prompt becomes hard to steer. A common
answer is a **supervisor** that delegates to **specialists**, each a small agent graph with its
own tools and prompt:

```text
                +----------------+
   START ---->  |   supervisor   | --- FINISH ---> final answer ---> END
                +----------------+
                   |          ^
        next worker|          | report
                   v          |
          records_worker / handbook_worker   (each is a subgraph from G3)
```

Two LangGraph mechanics: a specialist is a **compiled subgraph invoked from a node**, and the
supervisor node returns `Command(goto=...)` to choose the next node *and* update state in one
step. Two agentic lessons: multi-agent is a boundary decision (different tools, prompts,
permissions, owners), not a default; and it costs more model calls. Measure before you split.

### Step 1 — Two specialists and a structured supervisor decision

In [ ]:
records_worker_graph = build_agent([get_student, get_course], persona="You are the records desk. Report student and course facts with ids. Never quote policy.")   # ours (G3 helper)
handbook_worker_graph = build_agent([search_knowledge], persona="You are the handbook desk. Answer only from retrieved excerpts and cite the source.")

class SupervisorDecision(BaseModel):                       # ours, on Pydantic
    """Which specialist should work next, or FINISH when the reports answer the user."""
    next_worker: Literal["records_worker", "handbook_worker", "FINISH"]

def make_worker(name, graph):                              # ours: wrap a subgraph so its report is one named AIMessage
    def worker(state: ChatState):
        question = next(text_of(m) for m in reversed(state["messages"]) if isinstance(m, HumanMessage))
        result = graph.invoke({"messages": [HumanMessage(question)]})   # LangGraph: run the specialist on its own copy of the question
        return {"messages": [AIMessage(content=text_of(result["messages"][-1]), name=name)]}   # LangChain: name= labels the report
    return worker

SUPERVISOR_CALLS = []                                      # ours: one entry per supervisor decision (for Step 3)

def supervisor(state: ChatState):                          # ours: decides who works next
    SUPERVISOR_CALLS.append(1)
    reports = [m for m in state["messages"] if isinstance(m, AIMessage) and m.name]
    prompt = [SystemMessage("You supervise a records desk and a handbook desk. Read the user's question and the reports so far; choose the next desk, or FINISH when the reports answer the question.")] + state["messages"]
    decision = structured(SupervisorDecision).invoke(prompt)   # LangChain -> SupervisorDecision
    print(f"    supervisor: {len(reports)} report(s) so far -> {decision.next_worker}")
    return Command(goto="final" if decision.next_worker == "FINISH" else decision.next_worker)   # LangGraph: choose the next node from inside the node

def final(state: ChatState):                               # ours: one answer from all reports
    reply = model.invoke([SystemMessage("Combine the specialist reports into one short answer for the student.")] + state["messages"])   # LangChain
    return {"messages": [reply]}
print("specialists ready:", ["records_worker", "handbook_worker"])

### Step 2 — Wire the supervisor loop and run it

Note `destinations=` on the supervisor node: because it routes with `Command`, LangGraph cannot
see its edges from the code, so we declare the possible targets for drawing and validation.

In [ ]:
g = StateGraph(ChatState)
g.add_node("supervisor", supervisor, destinations=("records_worker", "handbook_worker", "final"))   # LangGraph: Command targets
g.add_node("records_worker", make_worker("records_worker", records_worker_graph))
g.add_node("handbook_worker", make_worker("handbook_worker", handbook_worker_graph))
g.add_node("final", final)
g.add_edge(START, "supervisor")
g.add_edge("records_worker", "supervisor")                 # LangGraph: every worker reports back to the supervisor
g.add_edge("handbook_worker", "supervisor")
g.add_edge("final", END)
campusai_v12 = g.compile()

question = "Student S001 wants to sit the CS201 exam. What is their attendance, and what does the handbook require?"
SUPERVISOR_CALLS.clear()
out = campusai_v12.invoke({"messages": [HumanMessage(question)]}, config={"recursion_limit": 12})   # LangGraph: bounded delegation
print()
show_messages(out["messages"])
print("\n" + campusai_v12.get_graph().draw_mermaid())

### Step 3 — Was the split worth it?

The single agent from G7 owns all the read tools and answers the same question. Count model
calls: the supervisor pattern paid for the supervisor's decisions plus each specialist's loop.

In [ ]:
single = build_agent(KNOWLEDGE_TOOLS)                      # ours: one agent, all tools
single_out = single.invoke({"messages": [HumanMessage(question)]})
count = lambda msgs: sum(1 for m in msgs if isinstance(m, AIMessage))   # ours: model calls = AI messages

records_calls = count(records_worker_graph.invoke({"messages": [HumanMessage(question)]})["messages"])    # rerun the workers to count their calls
handbook_calls = count(handbook_worker_graph.invoke({"messages": [HumanMessage(question)]})["messages"])
supervisor_calls = len(SUPERVISOR_CALLS) + 1               # every supervisor decision, plus the final answer
print("single agent      :", count(single_out["messages"]), "model calls")
print("supervisor pattern:", supervisor_calls + records_calls + handbook_calls, "model calls  (supervisor + final:", supervisor_calls, "| records:", records_calls, "| handbook:", handbook_calls, ")")

### Step 4 — The other pattern: handoffs

In the supervisor pattern one node stays in control. In a **handoff**, the *active agent changes*:
the records desk decides it cannot answer a policy question and transfers the conversation to the
handbook desk, which continues with the full history. The transfer is a tool the model may call;
the node turns that call into `Command(goto=...)`. Handoffs need no supervisor and fewer model
calls, at the price of less central control: each agent must know when to hand off.

```text
START -> records_agent --tool calls--> records_tools --> records_agent
              |  transfer_to_handbook
              v
         handbook_agent --tool calls--> handbook_tools --> handbook_agent --> END
```

In [ ]:
@tool
def transfer_to_handbook(reason: str) -> str:
    """Hand the conversation to the handbook desk when the question needs policy, rules or general information."""
    return "transferred"

def records_agent(state: ChatState):                        # ours: an agent node that can hand off
    reply = model.bind_tools([get_student, get_course, transfer_to_handbook]).invoke([SystemMessage("You are the records desk. Report student and course facts with ids. If the user also needs policy or rules, call transfer_to_handbook after your lookups.")] + state["messages"])   # LangChain
    transfers = [c for c in reply.tool_calls if c["name"] == "transfer_to_handbook"]
    if transfers:
        note = ToolMessage(content="Transferred to the handbook desk.", tool_call_id=transfers[0]["id"], name="transfer_to_handbook")   # LangChain: close the tool call
        return Command(goto="handbook_agent", update={"messages": [reply, note]})   # LangGraph: the active agent changes
    return Command(goto="records_tools" if reply.tool_calls else END, update={"messages": [reply]})

def handbook_agent(state: ChatState):                       # ours: continues with the whole history
    reply = model.bind_tools([search_knowledge]).invoke([SystemMessage("You are the handbook desk. The records desk transferred this conversation; use its findings and the handbook excerpts to answer, citing sources.")] + state["messages"])   # LangChain
    return Command(goto="handbook_tools" if reply.tool_calls else END, update={"messages": [reply]})

g = StateGraph(ChatState)
g.add_node("records_agent", records_agent, destinations=("records_tools", "handbook_agent", END))   # LangGraph: Command targets
g.add_node("records_tools", ToolNode([get_student, get_course]))
g.add_node("handbook_agent", handbook_agent, destinations=("handbook_tools", END))
g.add_node("handbook_tools", ToolNode([search_knowledge]))
g.add_edge(START, "records_agent")
g.add_edge("records_tools", "records_agent")
g.add_edge("handbook_tools", "handbook_agent")
handoff_graph = g.compile()

out = handoff_graph.invoke({"messages": [HumanMessage(question)]}, config={"recursion_limit": 12})
show_messages(out["messages"])
print("\nmodel calls with handoffs:", count(out["messages"]), "(supervisor pattern above:", supervisor_calls + records_calls + handbook_calls, ")")

### Recap

- **Problem seen:** one prompt with every tool becomes hard to steer and impossible to permission separately.
- **Layer added:** specialist subgraphs behind named worker nodes, a structured supervisor decision, `Command(goto=...)` routing, and a handoff tool that transfers control.
- **Evidence:** the supervisor delegated to both desks and finished; the handoff moved the conversation from one desk to the other with fewer model calls; the single agent was cheapest of all.